# examples-seen-step-axis — ex2: wandb.log payloads include examples_seen for cross-batch-size comparability

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `examples-seen-step-axis`. Running the final beacon cell reports progress against the `Trainer: examples-seen step axis` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: examples-seen step axis` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`examples-seen-step-axis`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "examples-seen-step-axis"
DD_SUBTOPIC = "Trainer: examples-seen step axis"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Logging `examples_seen` to wandb — quick refresher

The fair x-axis across runs with different batch sizes is `examples_seen = step * batch_size`. To make wandb actually USE that axis, you include it as a key in every `wandb.log(...)` call alongside the metrics — wandb's UI then lets you select it from the x-axis dropdown.

```python
for step, (x, y) in enumerate(loader, start=1):
    loss = train_one_batch(x, y)
    wandb.log({
        'loss':          loss.item(),
        'examples_seen': step * batch_size,
    })
```

Every payload contains the same `examples_seen` key so wandb knows which metric is the proposed x-axis. The `step=...` kwarg is OPTIONAL — by default wandb auto-increments its internal step with each call.

### Exercise 2 — wandb.log payloads include examples_seen for cross-batch-size comparability

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `wandb.log({'loss': ..., 'examples_seen': step * batch_size})` inside a training loop so every payload carries the fair cross-batch-size x-axis.
> Keywords: wandb, examples-seen, logging, x-axis
> ```

**KCs targeted:** `examples-seen-equals-step-times-batch-size`, `wandb-log-payload-contains-examples-seen-key`

Implement `ex2_train_with_wandb(losses, batch_size, wandb)`. A training-loop-like driver that emits one `wandb.log(...)` call per step, including the `examples_seen` key.

1. For `step, loss in enumerate(losses, start=1)`:
   - Compute `examples_seen = step * batch_size`.
   - Call `wandb.log({'loss': loss, 'examples_seen': examples_seen})`.
2. Return the total number of log calls made.

Inputs:
- `losses`: list of per-step float losses (the trainer's per-batch loss history).
- `batch_size`: int — examples per batch.
- `wandb`: a wandb-like object exposing `wandb.log(payload)`. The test passes a `MagicMock` instead of the real wandb.

Output: int — the number of log calls (= len(losses)).

**Critical:** EVERY payload must contain BOTH keys. The test inspects the mock's call args and asserts both `'loss'` and `'examples_seen'` appear in every dict passed to `wandb.log`.

In [ ]:
def ex2_train_with_wandb(losses, batch_size, wandb):
    n = 0
    for step, loss in enumerate(losses, start=1):
        examples_seen = step * batch_size
        wandb.log({'loss': loss, 'examples_seen': examples_seen})
        n += 1
    return n


<details><summary>Solution</summary>

```python
def ex2_train_with_wandb(losses, batch_size, wandb):
    n = 0
    for step, loss in enumerate(losses, start=1):
        examples_seen = step * batch_size
        wandb.log({'loss': loss, 'examples_seen': examples_seen})
        n += 1
    return n
```

**Why include `examples_seen` in EVERY payload (not just occasionally).** Wandb's x-axis selector picks one key globally and applies it to all metrics. If some payloads have `examples_seen` and others don't, the curve gets gaps. Easiest rule: include it in every `wandb.log` call alongside whatever you're actually logging.

**Why pass `wandb` as a parameter instead of `import wandb`.** Testability. The drill mocks `wandb`; a real trainer passes the live module. Same code path either way. ARENA's trainers wire wandb through the constructor for the same reason — easier to swap in a no-op logger for fast smoke tests.

**`step=` kwarg.** Wandb's `log` accepts a `step=` integer to override its internal step counter. Useful when YOUR step counter and wandb's drift apart (e.g. you log multiple metrics between optimizer steps). For most loops you can omit it.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()